# Binary Prediction of Poisonous Mushrooms

## Introduction

This project aims to predict whether a mushroom is edible or poisonous based on its physical characteristics, such as shape and color. Using machine learning, we will analyze the dataset and build a model to classify mushrooms accurately.

You can download the dataset and learn more about the competition here:  
[Playground Series - Season 4, Episode 8](https://www.kaggle.com/competitions/playground-series-s4e8/data)

In [3]:
import pandas as pd
import  numpy as np
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3116945 entries, 0 to 3116944
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   class                 object 
 2   cap-diameter          float64
 3   cap-shape             object 
 4   cap-surface           object 
 5   cap-color             object 
 6   does-bruise-or-bleed  object 
 7   gill-attachment       object 
 8   gill-spacing          object 
 9   gill-color            object 
 10  stem-height           float64
 11  stem-width            float64
 12  stem-root             object 
 13  stem-surface          object 
 14  stem-color            object 
 15  veil-type             object 
 16  veil-color            object 
 17  has-ring              object 
 18  ring-type             object 
 19  spore-print-color     object 
 20  habitat               object 
 21  season                object 
dtypes: float64(3), int64(1), object(18)
memory

In [6]:
train.head()

,id,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,0,e,8.80,f,s,u,f,a,c,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,a
1,1,p,4.51,x,h,o,f,a,c,n,...,NaN,y,o,NaN,NaN,t,z,NaN,d,w
2,2,e,6.94,f,s,b,f,x,c,w,...,NaN,s,n,NaN,NaN,f,f,NaN,l,w
3,3,e,3.88,f,y,g,f,s,NaN,g,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,u
4,4,e,5.85,x,l,w,f,d,NaN,w,...,NaN,NaN,w,NaN,NaN,f,f,NaN,g,a


In [7]:
test.head()

,id,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,3116945,8.64,x,NaN,n,t,NaN,NaN,w,11.13,...,b,NaN,w,u,w,t,g,NaN,d,a
1,3116946,6.90,o,t,o,f,NaN,c,y,1.27,...,NaN,NaN,n,NaN,NaN,f,f,NaN,d,a
2,3116947,2.00,b,g,n,f,NaN,c,n,6.18,...,NaN,NaN,n,NaN,NaN,f,f,NaN,d,s
3,3116948,3.47,x,t,n,f,s,c,n,4.98,...,NaN,NaN,w,NaN,n,t,z,NaN,d,u
4,3116949,6.17,x,h,y,f,p,NaN,y,6.73,...,NaN,NaN,y,NaN,y,t,NaN,NaN,d,u


In [8]:
train.shape

(3116945, 22)

In [9]:
train.isnull().sum()

id                            0
class                         0
cap-diameter                  4
cap-shape                    40
cap-surface              671023
cap-color                    12
does-bruise-or-bleed          8
gill-attachment          523936
gill-spacing            1258435
gill-color                   57
stem-height                   0
stem-width                    0
stem-root               2757023
stem-surface            1980861
stem-color                   38
veil-type               2957493
veil-color              2740947
has-ring                     24
ring-type                128880
spore-print-color       2849682
habitat                      45
season                        0
dtype: int64

In [10]:
miss_treshold = 500000
train= train.drop(columns = [col for col in train.columns if train[col].isnull().sum() > miss_treshold])
test= test.drop(test[['spore-print-color','veil-color','veil-type', 'stem-root', 'stem-surface','gill-attachment', 'gill-spacing','cap-surface']], axis=1)

In [11]:
# imputing missng values with "others"
train['cap-shape'] = train['cap-shape'].apply(lambda x: 'others' if x not in ['b', 'c', 'x', 'f', 'k', 's'] else x)
train['stem-color'] = train['stem-color'].apply(lambda x: 'others' if x not in ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'u', 'w', 'y'] else x)
train['does-bruise-or-bleed'] = train['does-bruise-or-bleed'].apply(lambda x: 'others' if x not in ['f', 't'] else x)
train['gill-color'] = train['gill-color'].apply(lambda x: 'others' if x not in ['k', 'n', 'b', 'h', 'g', 'r', 'o', 'p', 'u', 'e', 'w', 'y'] else x)
train['ring-type'] = train['ring-type'].apply(lambda x: 'others' if x not in ['c', 'e', 'f', 'l', 'n', 'p', 's', 'z'] else x)
train['has-ring'] = train['has-ring'].apply(lambda x: 'others' if x not in ['f', 't'] else x)
train['habitat'] = train['habitat'].apply(lambda x: 'others' if x not in ['g', 'l', 'm', 'p', 'u', 'd', 'w'] else x)
train['cap-color'] = train['cap-color'].apply(lambda x: 'others' if x not in ['n', 'b', 'c', 'g', 'r', 'p', 'u', 'e', 'w', 'y'] else x)


In [12]:
test['cap-shape'] = test['cap-shape'].apply(lambda x: 'others' if x not in ['b', 'c', 'x', 'f', 'k', 's'] else x)
test['stem-color'] = test['stem-color'].apply(lambda x: 'others' if x not in ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'u', 'w', 'y'] else x)
test['does-bruise-or-bleed'] = test['does-bruise-or-bleed'].apply(lambda x: 'others' if x not in ['f', 't'] else x)
test['gill-color'] = test['gill-color'].apply(lambda x: 'others' if x not in ['k', 'n', 'b', 'h', 'g', 'r', 'o', 'p', 'u', 'e', 'w', 'y'] else x)
test['ring-type'] = test['ring-type'].apply(lambda x: 'others' if x not in ['c', 'e', 'f', 'l', 'n', 'p', 's', 'z'] else x)
test['has-ring'] = test['has-ring'].apply(lambda x: 'others' if x not in ['f', 't'] else x)
test['habitat'] = test['habitat'].apply(lambda x: 'others' if x not in ['g', 'l', 'm', 'p', 'u', 'd', 'w'] else x)
test['cap-color'] = test['cap-color'].apply(lambda x: 'others' if x not in ['n', 'b', 'c', 'g', 'r', 'p', 'u', 'e', 'w', 'y'] else x)


In [13]:
# Replace cap-shape
test['cap-shape'] = test['cap-shape'].replace({
    "b": "bell", 
    "c": "conical", 
    "x": "convex", 
    "f": "flat", 
    "k": "knobbed", 
    "s": "sunken"
})



# Replace cap-color
test['cap-color'] = test['cap-color'].replace({
    "n": "brown", 
    "b": "buff", 
    "c": "cinnamon", 
    "g": "gray", 
    "r": "green", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace stem-color
test['stem-color'] = test['stem-color'].replace({
    "n": "brown", 
    "b": "buff", 
    "c": "cinnamon", 
    "g": "gray", 
    "o": "orange", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace does-bruise-or-bleed
test['does-bruise-or-bleed'] = test['does-bruise-or-bleed'].replace({
    "f": "no", 
    "t": "bruises"
})

# Replace gill-color
test['gill-color'] = test['gill-color'].replace({
    "k": "black", 
    "n": "brown", 
    "b": "buff", 
    "h": "chocolate", 
    "g": "gray", 
    "r": "green", 
    "o": "orange", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace ring-type
test['ring-type'] = test['ring-type'].replace({
    "c": "cobwebby", 
    "e": "evanescent", 
    "f": "flaring", 
    "l": "large", 
    "n": "none", 
    "p": "pendant", 
    "s": "sheathing", 
    "z": "zone"
})

# Replace has-ring
test['has-ring'] = test['has-ring'].replace({
    "f": "no", 
    "t": "yes"
})

# Replace habitat
test['habitat'] = test['habitat'].replace({
    "g": "grasses", 
    "l": "leaves", 
    "m": "meadows", 
    "p": "paths", 
    "u": "urban", 
    "d": "waste", 
    "w": "woods"
})


In [14]:
X = train.drop(["class"], axis=1)
y = train["class"]

In [15]:
# Replace cap-shape
train['cap-shape'] = train['cap-shape'].replace({
    "b": "bell", 
    "c": "conical", 
    "x": "convex", 
    "f": "flat", 
    "k": "knobbed", 
    "s": "sunken"
})


# Replace cap-color
train['cap-color'] = train['cap-color'].replace({
    "n": "brown", 
    "b": "buff", 
    "c": "cinnamon", 
    "g": "gray", 
    "r": "green", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace stem-color
train['stem-color'] = train['stem-color'].replace({
    "n": "brown", 
    "b": "buff", 
    "c": "cinnamon", 
    "g": "gray", 
    "o": "orange", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace does-bruise-or-bleed
train['does-bruise-or-bleed'] = train['does-bruise-or-bleed'].replace({
    "f": "no", 
    "t": "bruises"
})

# Replace gill-color
train['gill-color'] = train['gill-color'].replace({
    "k": "black", 
    "n": "brown", 
    "b": "buff", 
    "h": "chocolate", 
    "g": "gray", 
    "r": "green", 
    "o": "orange", 
    "p": "pink", 
    "u": "purple", 
    "e": "red", 
    "w": "white", 
    "y": "yellow"
})

# Replace ring-type
train['ring-type'] = train['ring-type'].replace({
    "c": "cobwebby", 
    "e": "evanescent", 
    "f": "flaring", 
    "l": "large", 
    "n": "none", 
    "p": "pendant", 
    "s": "sheathing", 
    "z": "zone"
})

# Replace has-ring
train['has-ring'] = train['has-ring'].replace({
    "f": "no", 
    "t": "yes"
})

# Replace habitat
train['habitat'] = train['habitat'].replace({
    "g": "grasses", 
    "l": "leaves", 
    "m": "meadows", 
    "p": "paths", 
    "u": "urban", 
    "d": "waste", 
    "w": "woods"
})


In [16]:
X = train.drop(["class"], axis=1)
y = train["class"].map({"e": 1, "p": 0})

In [17]:
#  label encoder
class FeatureEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.label_encoders = {}
        categorical_columns = X.select_dtypes(include=["object"]).columns
        for col in categorical_columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.label_encoders[col] = le
        return self

    def transform(self, X):
        for col, le in self.label_encoders.items():
            X[col] = le.transform(X[col])
        return X

In [18]:
# pipeline
pipeline = Pipeline([
    ("feature_encoder", FeatureEncoder()), 
])

In [19]:
train_df = pipeline.fit_transform(X)
test_df = pipeline.transform(test)

In [20]:
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size= 0.20, random_state=42)

In [21]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
test_df = scaler.transform(test_df)

In [22]:
# random classifier
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

RandomForestClassifier()

In [23]:
y_pred = rf.predict(X_test)
y_pred

array([0, 0, 1, ..., 0, 1, 0], dtype=int64)

In [24]:
score= accuracy_score(y_test,y_pred)
score

0.9822085407345975

In [25]:
# desicion tree
dct = DecisionTreeClassifier()
dct.fit(X_train, y_train)

DecisionTreeClassifier()

In [26]:
y_pred_dct = dct.predict(X_test)
y_pred_dct

array([0, 1, 1, ..., 0, 1, 0], dtype=int64)

In [27]:
score= accuracy_score(y_test,y_pred_dct)
score

0.9674793748365788

In [28]:
ans=rf.predict(test_df)
ans

array([1, 0, 0, ..., 0, 1, 1], dtype=int64)

In [59]:
submission = pd.DataFrame({"id": test["id"], "class": ans})  
submission.to_csv("mushroom_classification_submission1.csv", index=False)
print("saved")


saved



## Conclusion
In this project, we used machine learning to classify mushrooms as edible or poisonous. The Decision Tree model achieved an accuracy of 96.7%, while the Random Forest model performed better with an accuracy of 98.2%. The Random Forest model proved to be more effective, demonstrating the power of ensemble methods for this classification task.